# 代码实现记录


## 各模块


### def generate_token_dict(vocab) ->token_dict
vocab: a list of tokens    
Return : token_dict: dict : token -> int  
    使用enumerate()  

### def prepocess_input_sequence()
Args: input_str, token_dict, spc_tokens  
Return: out : list  
按照token_dict,将input_str划分为基本元素并分别映射到相应的整数   
得到out:由这些整数组成的列表  
    内部实现:使用字符串的split()  

### 数据处理流程
原始文本: 输入(问题):BOS ... EOS 输出(答案):BOS ... EOS  
在当前问题中,原始文本都是由基本元素组成的:数字0-9,BOS,EOS,POSITIVE,NEGATIVE,ADD,SUBSTRACT  
直接把它们当作vocab中的元素  
其中的非数字元素所组成的列表就是spc_tokens  
vocab: list  由上述基本元素构成  

vocab->generate_token_dict->token_dict  
input_str(原始文本)->prepocess_input_sequence->out

原问题被建模成序列到序列的问题,  
输入是问题表达式,输出是答案表达式,  
模型需要根据问题表达式,输出正确的答案表达式  


### scaled_dot_product_no_loop_batch()
    input : query,key,value,mask  
    output: y,weights_softmax  
    
    其中query,key,value是已经过投影的  
    形状要求都为(N,K,M)  
    (实际实现允许value的维度不同,只需query,key的相同) 

#### masked_fill(mask,)
    给定布尔张量,将True位置的元素设置为给定值
#### tril, triu
    将下三角/上三角部分保留,其余位置设置为0值(具体取值由数据类型决定)  
### SelfAttention: nn.Module
实际上并不要求是 自注意力,  
因为__forward__()中query,key,value由外界传入  
cross-attention可以直接使用该模块  
可以认为是SingleHeadAttention  

__init__(dim_in,dim_q,dim_v)  
    存储Q,K,V矩阵  
    Q,K:shape=(dim_in,dim_q)  
    V:shape=(dim_in,dim_v)  
__forward__(query,key,value,mask)->y (weighted sum of values)  
    返回values的加权求和,  
    并存储softmax生成的权重矩阵到self.weights_softmax  
    
    传入mask:无需为是否添加因果掩码而完成两份实现  

    内部调用了scaled_dot_product_no_loop_batch来实现注意力机制  


### MultiHeadAttention: nn.Module
使用SelfAttention实现  
__init__(num_heads,dim_in,dim_out)  
    num_heads: 注意力头的数量  
    dim_in: 词向量维度  
    **dim_out**: 各个注意力头的Q,K,V的输出维度  
    注意: num_heads*dim_out 才是该模块最终的输出维度  
__forward__(query,key,value,mask)->y (weighted sum of values)     
    与SelfAttention模块类似      

#### torch.cat
    输出使用torch.cat(tensor_list,dim=k)进行拼接  
    该方法接收一个存储张量的列表,  
    对列表中所有张量在维度k上进行拼接,并返回拼接后的张量    
    i.e. 若列表中每个张量x.shape=(N,K,dim),则拼接后张量形状为(N,K,dim*len(list))
    



### LayerNormalization: nn.Module
__init__(emb_dim,epsilon)    
    emb_dim: 词向量维度    
    epsilon:避免除以0(方差)    
__forward__(x)->y    

### FeedForwardBlock: nn.Module
Linear->ReLU->Linear  

__init__(inp_dim,hidden_dim_feedforward)  
    inp_dim:词向量维度  
    hidden_dim_feedforward: Linear的隐藏层维度;通常远高于inp_dim以引入参数  
__forward__(x)->y  

### EncoderBlock: nn.Module
架构:  
    sublayer: (1) MultiHeadAttention (2) FFN  
    input-> sublayer -> Add -> LN -> Dropout    
**注意**:  
虽然文档注释中说参考原论文实现,但实际上二者的Dropout顺序有差异  
作业中参考注释实现.  

原论文架构:
    input-> sublayer -> Dropout ->Add -> LN  


__init__(num_heads,emb_dim,feedforward_dim,dropout):  
    input,output的形状皆为:(N,K,M)  
    M = emb_dim,   
    emb_dim//num_heads = 单头内部的Q,K,V输出维度  

__forward__(x)->y  

### DecoderBlock: nn.Module
架构:  
    sublayer: (1) SelfAttention (2) CrossAttention (3) FFN  
    input -> sublayer -> Add -> LN -> Dropout  

__init__()  

__forward__()  

### Encoder
__init__(...,num_layers,...)  
    指明有num_layers个EncoderBlock即可  
__forward__(src_seq)-> src_seq的语义表示  

### Decoder
__init__(...,num_layers,...,vocab_len)  
    num_layers:指明DecoderBlock数量  
    vocab_len:词表大小;用于最终输出投影层  

__forward__(target_seq,enc_out,mask)  
    target_seq:decoder输入  
    enc_out:encoder输出    
    mask:自注意力的因果掩码

### Transformer
#### 数据
class AddSubDataset(torch.utils.data.Dataset)    
__getitem__()  
    dataset[i]获得的是:  
    索引为i的(preprocess_inp,inp_pos_enc,preprocess_out,out_pos_enc)   
    preprocess_inp/out为转化为整数ID后的问/答,1维张量,长为K/O  
    inp_pos_enc,out_pos_enc为位置编码,形状分别为(K,M)与(O,M)  

在Transformer内部forward时,才将整数ID序列转化为词嵌入向量,  
并加上位置编码.   
#### Transformer  
输入: 接受question,answer的tokens序列 (N,K)  
      与各自的位置掩码(K,K)  
      **note**:  
      一开始的文本中已包含BOS,EOS  
      answer作为decoder输入,  
      需要在嵌入前,除去最后的EOS.  
      
前向: 经过嵌入层后与位置掩码直接相加,得到词向量  
      词向量经过decoder得到logits (N,K,V)  
      为了后续方便计算交叉熵损失,    
      再将logits形状调整为(N*K,V)  
输出  logits (N*K,V)  (未经过softmax; reshaped)  
      

# 其余笔记


## 输入
token : 分词后得到的字符或者整数   
词向量(word vector/ embedding/dense embedding)  
sparse representation : one-hot vector 稀疏表示 通常独热编码只是理论上存在  
实践中通过Lookup查询:   
原始文本--tokenization(分词)-->token (整数ID或字符)--Lookup(查表操作)-->相应的词向量  

## 整体架构
一个transformer block:  
(1) 注意力块  
encoder:self-attention  
decoder:self-attention + cross-attention   
(2) FFN
且(1)/(2)的输入要做LN(pre-norm),并且与输出进行残差连接  
即: y = f(LN(h))+h, 其中f代表一个注意力块或者FFN  

encoder/decoder:由多个transformer block拼接而成  
decoder输出:transformer block的输出 + linear+softmax  

## attention
### encoder-decoder : Seq2Seq & MT(machine translation)

### attention as a general mechanism
给定query与values,以某种依赖于query的方式来加权组合values,得到输出  


### weighted_sum

非上下文表示(静态)x  
x->q,k,v  
Q K.T ->softmax ->得到权重分布  
->对V加权求和->softmax(Q @ K.T) V  

A.T @ B : (i,j)为A的i-th列向量与B的j-th列向量的内积  
A @ B.T : (i,j)为A的i-th行向量与B的j-th行向量的内积  

#### 一般设计权重的方法
n个值得到一组权重 -> a_i/∑_j a_j  求和为1  
n个值得到一组权重分布(权重和为1且非负) -> exp(a_i) / ∑_j exp(a_j)   i.e. softmax  

### FFN
(1)通过其中的非线性函数来引入逐元素的非线性  
(2)通常其隐藏层维度远大于隐向量维度,利用矩阵乘法的并行性趁机在此处引入大量参数,增强表示能力
